# Organized train/test datasets
Choose the combined dataset or either source. CSV `audio_path` values are relative to that dataset folder. All recordings use mono 16 kHz PCM16 audio.


In [ ]:
from pathlib import Path
import pandas as pd

root = next((parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
             if (parent / "data/karakalpak-combined/train.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("Run scripts/organize_datasets.py first")

# Alternatives: "karakalpak-audio-dataset-hf", "karakalpak-mendeley"
dataset_name = "karakalpak-combined"
dataset_dir = root / dataset_name
train = pd.read_csv(dataset_dir / "train.csv")
test = pd.read_csv(dataset_dir / "test.csv")
print(f"{len(train)} train / {len(test)} test")
train["duration"].sum() / 3600  # training audio hours


In [ ]:
train[["audio_path", "text", "duration"]].head()


In [ ]:
from datasets import Audio, Dataset, DatasetDict

frames = {"train": train, "test": test}
splits = {}
for name, frame in frames.items():
    frame = frame.copy()
    frame["audio"] = [{"path": str((dataset_dir / path).resolve()), "bytes": None}
                      for path in frame["audio_path"]]
    splits[name] = Dataset.from_pandas(frame, preserve_index=False).cast_column("audio", Audio(decode=False))
dataset = DatasetDict(splits)
dataset


## Listen to a training recording
The same row supplies the audio path and transcript.


In [ ]:
import IPython.display as ipd

sample = train.iloc[0]
print(sample["audio_path"])
print(sample["text"])
ipd.Audio(filename=str(dataset_dir / sample["audio_path"]))
